# DiffMM-CFM (Phương án 2: CFM loss weighting) — Colab Runner

Notebook này clone code từ **repo GitHub của chính bạn** (`DiffMM-CFM` — bản fork của
[HKUDS/DiffMM](https://github.com/HKUDS/DiffMM) đã có sẵn **Phương án 2** trong
[`Phuong_An_2_CFM_Loss_KeHoachChiTiet.md`](../Phuong_An_2_CFM_Loss_KeHoachChiTiet.md) được áp dụng
trực tiếp vào code — xem `DiffMM-CFM/README.md` để biết chi tiết), rồi tải dữ liệu từ Google Drive và
chạy huấn luyện.

> Tóm tắt thay đổi đã có sẵn trong repo: **giữ nguyên 100%** đường đi diffusion VP-style của DiffMM
> (không đổi sang OT như Phương án 1) và giữ nguyên mạng dự đoán α₀ trực tiếp (data-prediction), chỉ
> đổi **cách tính trọng số (weight) của loss huấn luyện** — từ công thức suy ra bằng ELBO/KL divergence
> (`w_ELBO(t) = SNR(t−1) − SNR(t)`) sang công thức suy trực tiếp từ CFM loss của Flow Matching
> (`w_CFM(t) = [μₜ' − (σₜ'/σₜ)·μₜ]²`, đã chứng minh + kiểm chứng số học rằng đây chính xác là dạng CFM
> loss thu gọn khi mạng vẫn dự đoán α₀). Toàn bộ phần còn lại — `q_sample`, `p_mean_variance`/
> `p_sample` (D4), MSI/`gc_loss`, top-k rebuild đồ thị, Cross-Modal Contrastive Augmentation,
> Multi-Modal Graph Aggregation, Multi-Task Training — **giữ nguyên 100%**.

**Trước khi chạy notebook này, bạn cần đẩy (push) `DiffMM-CFM` lên một repo GitHub RIÊNG của chính
bạn** (khuyến nghị dùng bản độc lập, không lồng trong repo project lớn, đã chuẩn bị sẵn tại
`E:\NAM_BA\DiffMM-CFM`) — xem hướng dẫn chi tiết ở `phuong_an_2_CFM_loss/README.md` (mục
"Bước 1 — Đẩy DiffMM-CFM/ lên GitHub riêng của bạn"). Sau đó dán URL repo đó vào `GITHUB_REPO_URL` ở
Cell 1 bên dưới.

**Cách dùng:** chạy lần lượt từng cell từ trên xuống. Cell đầu tiên là **nơi duy nhất** bạn cần chỉnh
(URL repo GitHub của bạn + link Google Drive chứa dữ liệu + chọn số epoch). Nhớ bật GPU:
`Runtime > Change runtime type > Hardware accelerator > GPU`.


## 1. Cell cấu hình đầu vào

Đây là **cell duy nhất bạn cần chỉnh sửa** trước khi chạy toàn bộ notebook (`Runtime > Run all`).

- `GITHUB_REPO_URL`: URL repo GitHub **của bạn** chứa folder `DiffMM-CFM` đã push lên (xem hướng dẫn
  push ở `phuong_an_2_CFM_loss/README.md`). Ví dụ: `https://github.com/<username>/DiffMM-CFM.git`.
- `GDRIVE_LINK`: link chia sẻ Google Drive tới dữ liệu dataset (chấp nhận **link file .zip** hoặc
  **link thư mục** — notebook sẽ tự nhận diện). File/thư mục cần chứa (ở đâu đó bên trong, không bắt
  buộc đúng cấu trúc thư mục tuyệt đối): `trnMat.pkl`, `tstMat.pkl`, `image_feat.npy`, `text_feat.npy`
  (và `audio_feat.npy` nếu là dataset `tiktok`) — đúng định dạng dữ liệu gốc của DiffMM.
- `DATASET_NAME`: phải là một trong `"tiktok"`, `"baby"`, `"sports"`.
- `NUM_EPOCHS`: số epoch huấn luyện — **chỉnh theo mong muốn của bạn** ở đây.
- `W_CLIP`: siêu tham số clip cho trọng số CFM (mặc định `50.0`, kiểu "Min-SNR weighting" — tránh
  trọng số "nổ" ở bước gần dữ liệu gốc, xem README của `DiffMM-CFM` để biết chi tiết).

In [ ]:
# ============================================================
# CELL 1 — CẤU HÌNH ĐẦU VÀO (chỗ DUY NHẤT bạn cần chỉnh sửa)
# ============================================================

GITHUB_REPO_URL = "https://github.com/thyelmot/DiffMM-CFM.git"  # repo GitHub riêng đã push code Phương án 2
GDRIVE_LINK = "https://drive.google.com/drive/folders/1UdSihFXvm5frxb3nAxJ_uMaoZrFssqqD?usp=sharing"  # link thư mục Google Drive chứa dữ liệu
DATASET_NAME = "tiktok"   # "tiktok" | "baby" | "sports"
NUM_EPOCHS = 50            # <-- chỉnh số epoch mong muốn ở đây
W_CLIP = 50.0              # [Phương án 2] clip cho trọng số CFM loss (Min-SNR style)

assert GITHUB_REPO_URL.strip() != "", "Hãy dán URL repo GitHub của bạn vào GITHUB_REPO_URL ở trên trước khi chạy tiếp."
assert GDRIVE_LINK.strip() != "", "Hãy dán link Google Drive vào biến GDRIVE_LINK ở trên trước khi chạy tiếp."
assert DATASET_NAME in ("tiktok", "baby", "sports"), "DATASET_NAME phải là 'tiktok', 'baby' hoặc 'sports'."
print(f"Cấu hình: repo={GITHUB_REPO_URL}, dataset={DATASET_NAME}, epochs={NUM_EPOCHS}, w_clip={W_CLIP}")

## 2. Cell setup môi trường

In [ ]:
# ============================================================
# CELL 2 — SETUP MÔI TRƯỜNG
# ============================================================
import torch

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "Chưa bật GPU cho Colab. Vào Runtime > Change runtime type > Hardware accelerator > GPU, "
    "rồi Runtime > Restart session và chạy lại từ Cell 1."
)

# Các thư viện repo gốc cần (numpy/scipy/torch Colab đã có sẵn) + các thư viện phụ trợ notebook cần thêm
!pip install -q gdown setproctitle tabulate

print("Đã cài đặt xong các thư viện cần thiết.")

## 3. Cell clone code

Clone từ **repo GitHub của chính bạn** (`GITHUB_REPO_URL` ở Cell 1) — repo này là bản fork của
[HKUDS/DiffMM](https://github.com/HKUDS/DiffMM) đã có sẵn patch Phương án 2 (xem
`phuong_an_2_CFM_loss/DiffMM-CFM/` cục bộ, và README của repo đó để biết cách push lên GitHub nếu bạn
chưa làm).

Cell này **luôn xoá bản clone cũ (nếu có) rồi clone lại từ đầu** — để chắc chắn mỗi lần chạy đều lấy
đúng phiên bản mới nhất trên GitHub, tránh tình trạng "đã sửa code trên GitHub rồi mà chạy vẫn lỗi y
hệt cũ" (do Colab giữ nguyên file từ lần clone trước trong cùng phiên chạy). Cell cũng **tự dò tìm
`Main.py`** trong toàn bộ cây thư mục vừa clone thay vì chỉ giả định nó nằm ngay gốc — vì một lỗi hay
gặp là repo trên GitHub bị **lồng thêm 1 cấp thư mục** (ví dụ do kéo-thả cả folder `DiffMM-CFM` lên qua
giao diện web "Upload files" thay vì push bằng lệnh `git`, khiến file thật sự nằm ở
`DiffMM-CFM/DiffMM-CFM/Main.py` thay vì `DiffMM-CFM/Main.py`). Nếu gặp đúng trường hợp này, cell sẽ tự
động chỉnh lại đường dẫn làm việc cho đúng, kèm cảnh báo để bạn biết.

In [ ]:
# ============================================================
# CELL 3 — CLONE CODE TỪ REPO GITHUB CỦA BẠN
# ============================================================
import glob
import os
import shutil

REPO_DIR = "DiffMM-CFM"

# Luôn xoá bản clone cũ (nếu có) rồi clone lại từ đầu — đảm bảo LUÔN lấy đúng code mới nhất
# trên GitHub. Nếu không làm vậy, chạy lại Cell 3 trong cùng 1 phiên Colab (mà không Restart
# runtime) sẽ bị bỏ qua bước clone và dùng nhầm code CŨ đã tải trước đó — rất dễ gây hiểu lầm
# "đã sửa lỗi rồi mà vẫn lỗi y hệt" khi chỉ mới cập nhật code trên GitHub chứ chưa clone lại.
if os.path.isdir(REPO_DIR):
    print(f"Xoá bản clone cũ '{REPO_DIR}' để lấy code MỚI NHẤT từ GitHub...")
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 {GITHUB_REPO_URL} {REPO_DIR}

assert os.path.isdir(REPO_DIR), (
    f"Không tìm thấy thư mục '{REPO_DIR}' sau khi clone — kiểm tra lại GITHUB_REPO_URL ở Cell 1 "
    "(repo phải công khai, hoặc bạn đã đăng nhập git trên Colab nếu là repo private)."
)

# Tự dò Main.py trong toàn bộ cây thư mục vừa clone, phòng trường hợp repo bị lồng thêm 1 cấp
# thư mục (ví dụ push nhầm cả folder cha, hoặc upload qua giao diện web GitHub thay vì git push).
main_py_candidates = glob.glob(os.path.join(REPO_DIR, "**", "Main.py"), recursive=True)
assert main_py_candidates, (
    f"Clone thành công nhưng KHÔNG tìm thấy Main.py ở đâu trong '{REPO_DIR}'.\n"
    f"Nội dung hiện có: {sorted(os.listdir(REPO_DIR))}\n"
    "Nhiều khả năng bạn đã trỏ GITHUB_REPO_URL sai repo, hoặc push nhầm nội dung "
    "(repo phải chứa đúng nội dung bên trong phuong_an_2_CFM_loss/DiffMM-CFM/, "
    "không phải cả project hay một folder cha khác)."
)
# Chọn ứng viên nông nhất (ít cấp thư mục nhất) nếu có nhiều Main.py
main_py_candidates.sort(key=lambda p: p.count(os.sep))
actual_dir = os.path.dirname(main_py_candidates[0])

if actual_dir != REPO_DIR:
    print(
        f"⚠ Main.py không nằm trực tiếp trong '{REPO_DIR}' mà nằm trong '{actual_dir}' "
        "— có thể repo bị lồng thêm 1 cấp thư mục. Tự động dùng đường dẫn này cho các bước sau."
    )
    REPO_DIR = actual_dir

print(f"\nREPO_DIR đang dùng: {REPO_DIR}")
print(sorted(os.listdir(REPO_DIR)))

## 4. Cell tải dữ liệu

Tự động tải dữ liệu từ `GDRIVE_LINK` (Cell 1) bằng `gdown`, tự giải nén (kể cả file `.zip` lồng bên
trong, ví dụ `image_feat.npy.zip` của dataset `baby`), rồi tự dò tìm thư mục chứa `trnMat.pkl` bất kể
cấu trúc thư mục bên trong file/folder Drive của bạn ra sao, và copy đúng vào
`DiffMM-CFM/Datasets/<DATASET_NAME>/` — đúng cấu trúc mà `DataHandler.py` gốc yêu cầu.

In [ ]:
# ============================================================
# CELL 4 — TẢI DỮ LIỆU TỪ GOOGLE DRIVE
# ============================================================
import glob
import shutil
import zipfile

import gdown

DATASETS_DIR = os.path.join(REPO_DIR, "Datasets")
TARGET_DIR = os.path.join(DATASETS_DIR, DATASET_NAME)
DOWNLOAD_DIR = "gdrive_download"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(TARGET_DIR, exist_ok=True)

if "/folders/" in GDRIVE_LINK:
    print("Phát hiện link THƯ MỤC Google Drive -> tải cả thư mục...")
    gdown.download_folder(url=GDRIVE_LINK, output=DOWNLOAD_DIR, quiet=False, use_cookies=False)
else:
    print("Phát hiện link FILE Google Drive -> tải file...")
    downloaded_path = gdown.download(
        url=GDRIVE_LINK, output=os.path.join(DOWNLOAD_DIR, "gdrive_data"), quiet=False, fuzzy=True
    )
    assert downloaded_path, "Tải dữ liệu từ Google Drive thất bại — kiểm tra lại link (phải ở chế độ chia sẻ công khai/Anyone with the link)."
    if downloaded_path.lower().endswith(".zip"):
        print(f"Giải nén {downloaded_path} ...")
        with zipfile.ZipFile(downloaded_path, "r") as zf:
            zf.extractall(DOWNLOAD_DIR)

# Tự giải nén mọi file .zip lồng bên trong (ví dụ image_feat.npy.zip của dataset baby)
for _ in range(3):  # vài vòng lặp để xử lý zip lồng nhiều cấp
    zip_files = glob.glob(os.path.join(DOWNLOAD_DIR, "**", "*.zip"), recursive=True)
    if not zip_files:
        break
    for zpath in zip_files:
        try:
            with zipfile.ZipFile(zpath, "r") as zf:
                zf.extractall(os.path.dirname(zpath))
            print(f"Đã giải nén: {zpath}")
            os.remove(zpath)
        except zipfile.BadZipFile:
            pass

# Tự dò tìm thư mục chứa trnMat.pkl, bất kể cấu trúc bên trong Drive của bạn ra sao
candidates = glob.glob(os.path.join(DOWNLOAD_DIR, "**", "trnMat.pkl"), recursive=True)
assert len(candidates) > 0, (
    "Không tìm thấy trnMat.pkl trong dữ liệu tải về từ Google Drive.\n"
    "Hãy kiểm tra: (1) link đã ở chế độ chia sẻ công khai (Anyone with the link) chưa, "
    "(2) dữ liệu có đủ trnMat.pkl, tstMat.pkl, image_feat.npy, text_feat.npy"
    + (", audio_feat.npy" if DATASET_NAME == "tiktok" else "") + " hay chưa."
)
src_dir = os.path.dirname(candidates[0])
print("Tìm thấy dữ liệu tại:", src_dir)

for fname in os.listdir(src_dir):
    fpath = os.path.join(src_dir, fname)
    if os.path.isfile(fpath):
        shutil.copy2(fpath, TARGET_DIR)

required = ["trnMat.pkl", "tstMat.pkl", "image_feat.npy", "text_feat.npy"]
if DATASET_NAME == "tiktok":
    required.append("audio_feat.npy")
missing = [f for f in required if not os.path.exists(os.path.join(TARGET_DIR, f))]
assert not missing, f"Thiếu file trong {TARGET_DIR}: {missing}. Kiểm tra lại dữ liệu trên Google Drive."

print(f"\nDữ liệu đã sẵn sàng tại: {TARGET_DIR}")
print(sorted(os.listdir(TARGET_DIR)))

## 5. Cell xác minh code đã có Phương án 2

Vì repo `DiffMM-CFM` bạn clone ở Cell 3 **đã có sẵn** patch Phương án 2 (áp dụng trực tiếp vào code,
không phải patch lúc chạy runtime nữa), cell này chỉ **kiểm tra lại cho chắc chắn** rằng bản clone
đúng là bản đã patch, trước khi bắt đầu huấn luyện. Nhắc lại các thay đổi đã có sẵn trong repo:

| File | Thay đổi |
|---|---|
| `Params.py` | Thêm argument mới `--w_clip` (mặc định `50.0`) — clip cho trọng số CFM loss |
| `Model.py` | Thêm class mới **`GaussianDiffusionCFM(GaussianDiffusion)`** ở cuối file — class `GaussianDiffusion` gốc **giữ nguyên không đổi**, chỉ override **đúng 1 hàm**: `training_losses` |
| `Main.py` | Đổi 1 dòng khởi tạo: `GaussianDiffusion(args.noise_scale, args.noise_min, args.noise_max, args.steps)` → `GaussianDiffusionCFM(args.noise_scale, args.noise_min, args.noise_max, args.steps, w_clip=args.w_clip)` |
| `DataHandler.py` | Fix tương thích scipy: `.A` → `.toarray()` (bẫy môi trường đã biết — `.A` bị gỡ ở scipy mới trên Colab) |

**Đối chiếu công thức gốc ↔ công thức mới** (chỉ khác cách tính trọng số, mọi thứ khác giữ nguyên):

| | DiffMM gốc (w_ELBO, eq 11-13) | Phương án 2 (w_CFM) |
|---|---|---|
| Target so sánh | `x̂₀` so với `x₀` (MSE) — **không đổi** | Y hệt |
| Công thức trọng số | `SNR(t−1) − SNR(t)`, `SNR(t)=ᾱₜ/(1−ᾱₜ)`, không clip | `[μₜ' − (σₜ'/σₜ)·μₜ]²`, có clip ở `w_clip` |
| Mạng dự đoán | α₀ (data-prediction) — **không đổi** | Y hệt |
| D4 (suy luận, `p_mean_variance`/`p_sample`) | Dùng `posterior_mean_coef1/2` (Bayes) — **không đổi** | Y hệt |

`q_sample` (D1), `p_mean_variance`/`p_sample` (D4), `SNR()`, MSI/`gc_loss` (D3) **giữ nguyên 100%** so
với code gốc — chỉ đúng dòng tính `weight` bên trong `training_losses` là khác.

In [ ]:
# ============================================================
# CELL 5 — XÁC MINH REPO ĐÃ CLONE CÓ ĐÚNG PATCH PHƯƠNG ÁN 2
# ============================================================

params_path = os.path.join(REPO_DIR, "Params.py")
model_path = os.path.join(REPO_DIR, "Model.py")
main_path = os.path.join(REPO_DIR, "Main.py")
datahandler_path = os.path.join(REPO_DIR, "DataHandler.py")

with open(params_path, "r", encoding="utf-8") as f:
    params_src = f.read()
with open(model_path, "r", encoding="utf-8") as f:
    model_src = f.read()
with open(main_path, "r", encoding="utf-8") as f:
    main_src = f.read()
with open(datahandler_path, "r", encoding="utf-8") as f:
    datahandler_src = f.read()

checks = {
    "Params.py có --w_clip": "--w_clip" in params_src,
    "Model.py có class GaussianDiffusionCFM": "class GaussianDiffusionCFM" in model_src,
    "Main.py import GaussianDiffusionCFM": "GaussianDiffusionCFM" in main_src and "from Model import" in main_src,
    "Main.py khởi tạo bằng GaussianDiffusionCFM (không phải GaussianDiffusion gốc)":
        "GaussianDiffusionCFM(args.noise_scale" in main_src,
    "DataHandler.py đã fix scipy .A -> .toarray()": ".toarray()" in datahandler_src and ".trnMat.A" not in datahandler_src,
}

for name, ok in checks.items():
    print(("✓ " if ok else "✗ ") + name)

assert all(checks.values()), (
    "Repo vừa clone KHÔNG có đúng patch Phương án 2. Kiểm tra lại: bạn đã push đúng folder "
    "phuong_an_2_CFM_loss/DiffMM-CFM (chứ không phải clone nhầm HKUDS/DiffMM gốc) chưa, "
    "và GITHUB_REPO_URL ở Cell 1 có trỏ đúng repo/branch đó không."
)
print("\n--- Repo đã clone đúng là bản có Phương án 2 (CFM loss weighting) ---")

## 6. Cell chạy huấn luyện

Dùng đúng bộ siêu tham số khuyến nghị theo dataset trong `README.md` gốc của DiffMM (chỉ thay `--epoch`
bằng `NUM_EPOCHS` bạn đã chọn ở Cell 1, và thêm `--w_clip`). Chạy bằng `subprocess` (thay vì
`!cd ... && ... | tee ...`) để tránh lỗi mơ hồ về đường dẫn tương đối giữa các cell, và để **báo lỗi
ngay tại đây** (thay vì để Cell 7 báo lỗi khó hiểu) nếu `Main.py` thoát với lỗi.

In [ ]:
# ============================================================
# CELL 6 — CHẠY HUẤN LUYỆN
# ============================================================
import subprocess

DATASET_HP = {
    "tiktok": ["--reg", "1e-4", "--ssl_reg", "1e-2", "--trans", "1", "--e_loss", "0.1", "--cl_method", "1"],
    "baby":   ["--reg", "1e-5", "--ssl_reg", "1e-1", "--keepRate", "1", "--e_loss", "0.01"],
    "sports": ["--reg", "1e-6", "--ssl_reg", "1e-2", "--temp", "0.1", "--ris_lambda", "0.1", "--e_loss", "0.5", "--keepRate", "1", "--trans", "1"],
}

# LOG_PATH dùng đường dẫn TUYỆT ĐỐI, tính trước khi đổi thư mục làm việc cho tiến trình con,
# để Cell 7 (và cả việc bạn tự mở file lên xem) luôn tìm đúng file log, không phụ thuộc cwd hiện tại.
LOG_PATH = os.path.abspath("train_log.txt")

cmd = [
    "python", "Main.py",
    "--data", DATASET_NAME,
    "--epoch", str(NUM_EPOCHS),
    "--w_clip", str(W_CLIP),
] + DATASET_HP[DATASET_NAME]

print("Lệnh chạy:", " ".join(cmd))
print("Thư mục làm việc của Main.py:", os.path.abspath(REPO_DIR))
print("Log sẽ được ghi vào:", LOG_PATH)
print()

with open(LOG_PATH, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, universal_newlines=True,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    process.wait()

print(f"\n\nMain.py kết thúc với exit code: {process.returncode}")
assert process.returncode == 0, (
    f"Main.py thoát với lỗi (exit code {process.returncode}). Xem log phía trên (hoặc mở file "
    f"{LOG_PATH}) để biết chi tiết lỗi — sửa xong thì chạy lại Cell 6 này trước khi sang Cell 7."
)
assert os.path.exists(LOG_PATH) and os.path.getsize(LOG_PATH) > 0, f"Không tạo được file log tại {LOG_PATH}."
print(f"Huấn luyện xong, đã ghi log đầy đủ vào: {LOG_PATH}")

## 7. Cell xuất kết quả

Đọc log huấn luyện vừa chạy ở Cell 6, tìm dòng `Best epoch : ... , Recall : ... , NDCG : ... ,
Precision ...` (được `Main.py` gốc tự in ra sau khi huấn luyện xong) và trình bày lại thành bảng dễ
đọc. Nếu bạn thấy lỗi `FileNotFoundError` ở đây, gần như chắc chắn là do **Cell 6 chưa được chạy
(hoặc chạy chưa xong) trong phiên hiện tại** — ví dụ Colab bị ngắt kết nối/reset runtime giữa chừng
khiến file tạm bị xoá — chỉ cần chạy lại Cell 6 (không cần chạy lại từ đầu, trừ khi Cell 3/4 cũng báo
thiếu) rồi chạy lại Cell 7.

In [ ]:
# ============================================================
# CELL 7 — XUẤT KẾT QUẢ (các chỉ số tốt nhất đạt được)
# ============================================================
import re

import pandas as pd

assert "LOG_PATH" in globals() and os.path.exists(LOG_PATH), (
    "Không tìm thấy file log huấn luyện (biến LOG_PATH chưa có hoặc file không tồn tại). "
    "Nguyên nhân thường gặp nhất: Cell 6 chưa được chạy trong phiên này, hoặc phiên Colab đã bị "
    "reset/ngắt kết nối (mất hết file tạm) giữa lúc chạy Cell 6 và Cell 7. "
    "=> Hãy chạy lại Cell 6, đợi huấn luyện xong hẳn, rồi chạy lại Cell 7 này."
)

with open(LOG_PATH, "r", encoding="utf-8") as f:
    log_text = f.read()

m = re.search(
    r"Best epoch\s*:\s*(\d+)\s*,\s*Recall\s*:\s*([\d.]+)\s*,\s*NDCG\s*:\s*([\d.]+)\s*,\s*Precision\s*([\d.]+)",
    log_text,
)
assert m, (
    f"Cell 6 đã chạy xong (log tồn tại tại {LOG_PATH}) nhưng không tìm thấy dòng 'Best epoch ...' "
    "trong log — mở file này lên để xem Main.py báo lỗi gì (ví dụ thiếu dữ liệu, sai đường dẫn, "
    "out of memory...), sửa xong thì chạy lại Cell 6."
)

best_epoch, recall, ndcg, precision = m.groups()

result_df = pd.DataFrame([{
    "Dataset": DATASET_NAME,
    "Phương án": "Phương án 2 (CFM loss weighting)",
    "Best Epoch": int(best_epoch),
    "Recall@20": float(recall),
    "NDCG@20": float(ndcg),
    "Precision@20": float(precision),
    "NUM_EPOCHS": NUM_EPOCHS,
    "W_CLIP": W_CLIP,
}])

display(result_df)
print()
print(result_df.to_markdown(index=False))